# Lantern candidate cutout viewer — v1

This notebook (previous name `lantern_candidate_cutout_viewer_v1-3-Copy1`) is deliberately separated from the candidate-selection notebook.

**Input:** the final candidate list written by  
`lantern_candidate_rgb_vetting_hybrid_v7.ipynb`

**Purpose:** visual inspection only.

For each candidate it can show:

1. **DP2 IRG** deep-coadd color cutout;
2. **DP2 ZIR** deep-coadd color cutout;
3. **Legacy Survey DR11** JPEG cutout;
4. latest available ANTARES/Rubin alert **visit / science** thumbnail;
5. latest available ANTARES/Rubin alert **template** thumbnail;
6. latest available ANTARES/Rubin alert **difference** thumbnail.

The figure also prints the candidate quality quantities, including the
**Second-Pass score**.

The number of candidates shown is controlled by `START` and `N_SHOW` in
Section 1. Increase `N_SHOW` to 100 or more when ready. To make that practical, all remote images are cached
on disk and DP2 requests use deliberate throttling plus HTTP-429-aware retries.

### Data sources

- DP2 deep-coadd cutouts: Rubin RSP SIA/SODA.
- Legacy Survey cutout: official DR11 `jpeg-cutout` service.
- Alert thumbnails: `antares_client.search.get_thumbnails(alert_id)`.

ANTARES documentation explicitly supports `get_thumbnails("lsst:<alert_id>")`
for LSST alerts.

In [ ]:
import ast
import base64
import gzip
import io
import json
import random
import re
import time
from collections.abc import Mapping
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.visualization import (
    AsinhStretch,
    ImageNormalize,
    LinearStretch,
    PercentileInterval,
    ZScaleInterval,
)
from IPython.display import display, Markdown
from PIL import Image

from lsst.images.serialization import read_archive
from lsst.rsp import RSPDiscovery
from lsst.rsp.utils import get_pyvo_auth
from pyvo.dal.adhoc import SodaQuery

try:
    from antares_client.search import get_thumbnails
except ImportError as e:
    raise ImportError(
        "This notebook needs antares-client. "
        "Install/upgrade it in the notebook environment, e.g. "
        "`pip install --upgrade antares-client`."
    ) from e

%matplotlib inline

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

## 1. Inputs and controls

In [ ]:
# ============================================================
# INPUTS
# ============================================================

# Output from lantern_candidate_rgb_vetting_hybrid_v7.ipynb
CANDIDATES_FILE = Path(
    "candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut/"
    "second_pass_candidates_gaia_bp_rp_ge_2p5.parquet"
)

# Needed only to identify the most recent alert(s) for each target.
SECOND_PASS_DIR = Path(
    "second_pass_output_hybrid_multi_loci"
)

CANDIDATE_ALERTS_FILE = (
    SECOND_PASS_DIR
    / "second_pass_candidate_alerts.parquet"
)

# ============================================================
# WHICH CANDIDATES TO SHOW
# ============================================================

START = 0
N_SHOW = 1800

# Sort the final candidate list before taking START:N_SHOW.
SORT_BY = "second_pass_score"
SORT_ASCENDING = False

# ============================================================
# OUTPUT
# ============================================================

OUTPUT_DIR = Path(
    "lantern_candidate_rgb_vetting_gaia_bp_rp_ge_2p5"
)

PDF_DIR = OUTPUT_DIR / "pdf"
CACHE_DIR = OUTPUT_DIR / "cache"

DP2_CACHE_DIR = CACHE_DIR / "dp2"
LEGACY_CACHE_DIR = CACHE_DIR / "legacy"
ANTARES_CACHE_DIR = CACHE_DIR / "antares"

for directory in [
    OUTPUT_DIR,
    PDF_DIR,
    DP2_CACHE_DIR,
    LEGACY_CACHE_DIR,
    ANTARES_CACHE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

#SHOW_IN_NOTEBOOK = True
SHOW_IN_NOTEBOOK = False

SAVE_PDF = True
OVERWRITE_PDF = True

# ============================================================
# DP2 CUTOUT SETTINGS
# ============================================================

# 0.001 deg radius = 3.6 arcsec radius = 7.2 arcsec full width.
CUTOUT_RADIUS = 0.001

DP2_ASINH_A = 0.002
DP2_VMIN = -0.03
DP2_VMAX = 3000

# Slow, serial requests are intentional. This is much safer for ~100 targets.
DP2_REQUEST_PAUSE = 1.5
MAX_RETRIES = 6
DEFAULT_RETRY_SECONDS = 15

# ============================================================
# LEGACY SURVEY SETTINGS
# ============================================================

LEGACY_LAYER = "ls-dr11"
LEGACY_PIXSCALE = 0.262
LEGACY_SIZE = 80
LEGACY_REQUEST_PAUSE = 0.25

# ============================================================
# ANTARES THUMBNAIL SETTINGS
# ============================================================

# Try the newest alert first. If it has no usable thumbnails, try a few older ones.
MAX_RECENT_ALERTS_TO_TRY = 3
ANTARES_REQUEST_PAUSE = 0.5

# ============================================================
# FIGURE
# ============================================================

FIGSIZE = (11, 8)
#PNG_DPI = 150

## 2. Load the final candidate list and alert IDs

In [ ]:
candidates = pd.read_parquet(
    CANDIDATES_FILE
).copy()

candidate_alerts = pd.read_parquet(
    CANDIDATE_ALERTS_FILE
).copy()

required_candidates = {
    "target_id",
    "ra",
    "dec",
    "second_pass_score",
}

missing = (
    required_candidates
    - set(candidates.columns)
)

if missing:
    raise KeyError(
        "The final candidate list is missing required columns: "
        f"{sorted(missing)}"
    )

required_alerts = {
    "target_id",
    "locus_id",
    "alert_id",
    "mjd",
}

missing = (
    required_alerts
    - set(candidate_alerts.columns)
)

if missing:
    raise KeyError(
        "The candidate-alert table is missing required columns: "
        f"{sorted(missing)}"
    )

candidates = (
    candidates
    .sort_values(
        SORT_BY,
        ascending=SORT_ASCENDING,
    )
    .reset_index(drop=True)
)

# Viewer rank refers to the final post-vetting candidate list.
candidates["viewer_rank"] = (
    np.arange(
        len(candidates)
    )
    + 1
)

candidate_ids = set(
    candidates["target_id"]
)

candidate_alerts = (
    candidate_alerts[
        candidate_alerts["target_id"].isin(
            candidate_ids
        )
    ]
    .sort_values(
        [
            "target_id",
            "mjd",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

subset = candidates.iloc[
    START:START + N_SHOW
].copy()

print(
    f"Final candidates loaded: {len(candidates):,}"
)

print(
    f"Viewer subset: rows {START} to "
    f"{START + len(subset) - 1} "
    f"(N={len(subset)})"
)

preview_columns = [
    c
    for c in [
        "viewer_rank",
        "target_id",
        "ra",
        "dec",
        "second_pass_score",
        "first_cut_max_score",
        "n_tagged",
        "g_mag",
        "r_mag",
        "i_mag",
        "z_mag",
        "gaia_bp_rp_max",
        "extendedness_median",
        "ap_psf_abs_fracdiff_median",
        "delta_e_median",
        "abs_delta_e_median",
        "n_lsst_alerts",
        "gaia_in_qso_candidates_any",
        "milliquas_match",
        "known_lens_match",
        "known_lens_name",
        "known_lens_sep_arcsec",
    ]
    if c in subset.columns
]

display(
    subset[
        preview_columns
    ]
)

## 3. Generic retry and cache helpers

In [ ]:
def safe_filename(value):
    text = str(value)
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        text,
    )


def retry_seconds_from_error(
    error,
    default=DEFAULT_RETRY_SECONDS,
):
    text = str(error)

    patterns = [
        r"Retry after\s+(\d+(?:\.\d+)?)\s+seconds",
        r"Retry-After[:=\s]+(\d+(?:\.\d+)?)",
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )

        if match:
            return float(
                match.group(1)
            )

    return float(default)


def call_with_429_retry(
    func,
    label,
    max_retries=MAX_RETRIES,
):
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            return func()

        except Exception as error:
            text = str(error).lower()

            is_rate_limit = (
                "429" in text
                or "rate limit" in text
            )

            if not is_rate_limit:
                raise

            if attempt >= max_retries:
                raise

            wait = (
                retry_seconds_from_error(
                    error
                )
                + random.uniform(
                    1.0,
                    3.0,
                )
            )

            print(
                f"{label}: HTTP 429 / rate limit. "
                f"Waiting {wait:.1f} s "
                f"before retry {attempt + 1}/{max_retries}."
            )

            time.sleep(
                wait
            )


def save_npy(
    path,
    array,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.save(
        path,
        np.asarray(array),
        allow_pickle=False,
    )


def load_npy(
    path,
):
    return np.load(
        path,
        allow_pickle=False,
    )

## 4. DP2 IRG and ZIR helpers

In [ ]:
discovery = RSPDiscovery(
    "dp2"
)

sia_client = (
    discovery
    .get_sia_client()
)

# Memory cache for SIA searches during this kernel session.
if "SIA_RESULTS_CACHE" not in globals():
    SIA_RESULTS_CACHE = {}


def dp2_cache_path(
    target_id,
    band,
):
    return (
        DP2_CACHE_DIR
        / (
            f"{safe_filename(target_id)}"
            f"_{band}.npy"
        )
    )


def get_sia_results(
    target_id,
    ra,
    dec,
    radius=CUTOUT_RADIUS,
):
    key = (
        str(target_id),
        round(
            float(ra),
            8,
        ),
        round(
            float(dec),
            8,
        ),
        round(
            float(radius),
            8,
        ),
    )

    if key in SIA_RESULTS_CACHE:
        return SIA_RESULTS_CACHE[key]

    def _search():
        return sia_client.search(
            pos=(
                float(ra),
                float(dec),
                float(radius),
            ),
            calib_level=3,
            dpsubtype="lsst.deep_coadd",
        )

    results = call_with_429_retry(
        _search,
        label=f"SIA search {target_id}",
    )

    SIA_RESULTS_CACHE[key] = results

    return results


def get_dp2_band_array(
    target_id,
    ra,
    dec,
    band,
    radius=CUTOUT_RADIUS,
):
    cache_path = dp2_cache_path(
        target_id,
        band,
    )

    if cache_path.exists():
        return load_npy(
            cache_path
        )

    results = get_sia_results(
        target_id,
        ra,
        dec,
        radius=radius,
    )

    band_arr = np.asarray(
        results["lsst_band"]
    ).astype(str)

    indices = np.where(
        band_arr == band
    )[0]

    if len(indices) == 0:
        print(
            f"{target_id}: no DP2 {band}-band deep coadd."
        )
        return None

    index = int(
        indices[-1]
    )

    def _get_datalink():
        return discovery.get_datalink_results(
            results[index]
        )

    dl_result = call_with_429_retry(
        _get_datalink,
        label=f"DataLink {target_id} {band}",
    )

    def _get_cutout():
        query = SodaQuery.from_resource(
            dl_result,
            dl_result.get_adhocservice_by_id(
                "cutout-sync"
            ),
            session=get_pyvo_auth(),
        )

        query.circle = (
            float(ra) * u.deg,
            float(dec) * u.deg,
            float(radius) * u.deg,
        )

        cutout_bytes = (
            query
            .execute_stream()
            .read()
        )

        query.raise_if_error()

        return read_archive(
            io.BytesIO(
                cutout_bytes
            )
        )

    cutout = call_with_429_retry(
        _get_cutout,
        label=f"SODA cutout {target_id} {band}",
    )

    array = np.asarray(
        cutout.array,
        dtype=float,
    )

    save_npy(
        cache_path,
        array,
    )

    time.sleep(
        DP2_REQUEST_PAUSE
    )

    return array


def normalize_dp2_array(
    array,
):
    norm = ImageNormalize(
        array,
        vmin=DP2_VMIN,
        vmax=DP2_VMAX,
        stretch=AsinhStretch(
            a=DP2_ASINH_A
        ),
        clip=True,
    )

    return norm(
        array
    )


def make_dp2_rgb(
    red_array,
    green_array,
    blue_array,
):
    if any(
        array is None
        for array in [
            red_array,
            green_array,
            blue_array,
        ]
    ):
        return None

    return np.dstack(
        [
            normalize_dp2_array(
                red_array
            ),
            normalize_dp2_array(
                green_array
            ),
            normalize_dp2_array(
                blue_array
            ),
        ]
    )


def get_dp2_irg_zir(
    row,
):
    target_id = row["target_id"]
    ra = float(row["ra"])
    dec = float(row["dec"])

    arrays = {}

    for band in [
        "g",
        "r",
        "i",
        "z",
    ]:
        try:
            arrays[band] = get_dp2_band_array(
                target_id,
                ra,
                dec,
                band,
            )
        except Exception as error:
            print(
                f"{target_id}: failed DP2 {band}-band cutout: {error}"
            )
            arrays[band] = None

    # IRG -> R=i, G=r, B=g
    irg = make_dp2_rgb(
        arrays["i"],
        arrays["r"],
        arrays["g"],
    )

    # ZIR -> R=z, G=i, B=r
    zir = make_dp2_rgb(
        arrays["z"],
        arrays["i"],
        arrays["r"],
    )

    return (
        irg,
        zir,
    )

## 5. Legacy Survey DR11 cutout helper

In [ ]:
def legacy_viewer_url(
    ra,
    dec,
):
    return (
        "https://www.legacysurvey.org/viewer"
        f"?ra={float(ra):.8f}"
        f"&dec={float(dec):.8f}"
        f"&layer={LEGACY_LAYER}"
        "&zoom=16"
    )


def legacy_cutout_url(
    ra,
    dec,
):
    return (
        "https://www.legacysurvey.org/viewer/jpeg-cutout"
        f"?ra={float(ra):.8f}"
        f"&dec={float(dec):.8f}"
        f"&layer={LEGACY_LAYER}"
        f"&pixscale={LEGACY_PIXSCALE}"
        f"&size={int(LEGACY_SIZE)}"
    )


def get_legacy_cutout(
    row,
):
    target_id = row["target_id"]

    cache_path = (
        LEGACY_CACHE_DIR
        / (
            f"{safe_filename(target_id)}.jpg"
        )
    )

    if cache_path.exists():
        with Image.open(
            cache_path
        ) as image:
            return np.asarray(
                image.convert("RGB")
            )

    url = legacy_cutout_url(
        row["ra"],
        row["dec"],
    )

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):
        response = requests.get(
            url,
            timeout=60,
        )

        if response.status_code == 200:
            cache_path.write_bytes(
                response.content
            )

            time.sleep(
                LEGACY_REQUEST_PAUSE
            )

            with Image.open(
                io.BytesIO(
                    response.content
                )
            ) as image:
                return np.asarray(
                    image.convert("RGB")
                )

        if response.status_code == 429:
            if attempt >= MAX_RETRIES:
                response.raise_for_status()

            retry_after = response.headers.get(
                "Retry-After",
                DEFAULT_RETRY_SECONDS,
            )

            try:
                wait = float(
                    retry_after
                )
            except Exception:
                wait = float(
                    DEFAULT_RETRY_SECONDS
                )

            wait += random.uniform(
                1.0,
                3.0,
            )

            print(
                f"Legacy Survey {target_id}: HTTP 429. "
                f"Waiting {wait:.1f} s."
            )

            time.sleep(
                wait
            )

            continue

        response.raise_for_status()

    return None

## 6. ANTARES recent alert thumbnails

In [ ]:
def as_mapping(
    value,
):
    if isinstance(
        value,
        Mapping,
    ):
        return dict(value)

    if isinstance(
        value,
        str,
    ):
        for parser in [
            json.loads,
            ast.literal_eval,
        ]:
            try:
                parsed = parser(
                    value
                )

                if isinstance(
                    parsed,
                    Mapping,
                ):
                    return dict(
                        parsed
                    )
            except Exception:
                pass

    return {}


def get_recent_alert_rows(
    target_id,
    n=MAX_RECENT_ALERTS_TO_TRY,
):
    return (
        candidate_alerts[
            candidate_alerts["target_id"]
            == target_id
        ]
        .sort_values(
            "mjd",
            ascending=False,
        )
        .head(n)
        .copy()
    )


def thumbnail_item_to_array(item):
    if item is None:
        return None

    if not isinstance(item, dict):
        return None

    blob = item.get("blob")

    if blob is None:
        return None

    # Convert blob to bytes
    if isinstance(blob, memoryview):
        blob = blob.tobytes()

    elif isinstance(blob, bytearray):
        blob = bytes(blob)

    elif isinstance(blob, str):
        # Handle possible data URL
        if blob.startswith("data:"):
            blob = blob.split(",", 1)[1]

        blob = base64.b64decode(blob)

    # Let PIL determine the image format automatically
    try:
        with Image.open(
            io.BytesIO(blob)
        ) as image:
            return np.asarray(
                image.convert("RGB")
            )

    except Exception as e:
        print(
            f"Could not decode ANTARES thumbnail: {e}"
        )
        return None


def antares_cache_path(
    alert_id,
    kind,
):
    return (
        ANTARES_CACHE_DIR
        / (
            f"{safe_filename(alert_id)}"
            f"_{kind}.npy"
        )
    )


def canonical_thumbnail_item(
    thumbnails,
    kind,
):
    if not isinstance(
        thumbnails,
        Mapping,
    ):
        return None

    key_options = {
        "template": [
            "template",
            "Template",
            "cutoutTemplate",
        ],
        "science": [
            "science",
            "Science",
            "visit",
            "Visit",
            "cutoutScience",
        ],
        "difference": [
            "difference",
            "Difference",
            "diff",
            "cutoutDifference",
        ],
    }

    for key in key_options[
        kind
    ]:
        if key in thumbnails:
            return thumbnails[
                key
            ]

    return None


def get_antares_thumbnail_arrays_for_alert(
    alert_id,
):
    kinds = [
        "template",
        "science",
        "difference",
    ]

    cached = {}
    all_cached = True

    for kind in kinds:
        path = antares_cache_path(
            alert_id,
            kind,
        )

        if path.exists():
            cached[kind] = load_npy(
                path
            )
        else:
            cached[kind] = None
            all_cached = False

    if all_cached:
        return cached

    def _fetch():
        return get_thumbnails(
            str(alert_id)
        )

    thumbnails = call_with_429_retry(
        _fetch,
        label=f"ANTARES thumbnails {alert_id}",
    )

    time.sleep(
        ANTARES_REQUEST_PAUSE
    )

    arrays = {}

    for kind in kinds:
        if cached[
            kind
        ] is not None:
            arrays[kind] = cached[
                kind
            ]
            continue

        item = canonical_thumbnail_item(
            thumbnails,
            kind,
        )

        try:
            array = thumbnail_item_to_array(
                item
            )
        except Exception as error:
            print(
                f"{alert_id}: could not decode ANTARES "
                f"{kind} thumbnail: {error}"
            )
            array = None

        arrays[kind] = array

        if array is not None:
            save_npy(
                antares_cache_path(
                    alert_id,
                    kind,
                ),
                array,
            )

    return arrays


def get_recent_antares_thumbnails(
    target_id,
):
    recent = get_recent_alert_rows(
        target_id
    )

    if len(recent) == 0:
        return (
            None,
            {
                "template": None,
                "science": None,
                "difference": None,
            },
        )

    for _, alert_row in recent.iterrows():
        alert_id = str(
            alert_row["alert_id"]
        )

        # We expect LSST alert IDs in the form "lsst:<id>".
        if not alert_id.startswith(
            "lsst:"
        ):
            continue

        try:
            arrays = (
                get_antares_thumbnail_arrays_for_alert(
                    alert_id
                )
            )
        except Exception as error:
            print(
                f"{target_id}: failed ANTARES thumbnails "
                f"for {alert_id}: {error}"
            )
            continue

        if any(
            array is not None
            for array in arrays.values()
        ):
            meta = {
                "alert_id": alert_id,
                "mjd": alert_row.get(
                    "mjd",
                    np.nan,
                ),
                "locus_id": alert_row.get(
                    "locus_id",
                    "",
                ),
            }

            return (
                meta,
                arrays,
            )

    return (
        None,
        {
            "template": None,
            "science": None,
            "difference": None,
        },
    )

## 7. Thumbnail display normalization

In [ ]:
def display_gray_image(
    ax,
    array,
    title,
    mode="asinh",
):
    ax.set_title(
        title
    )

    ax.set_xticks([])
    ax.set_yticks([])

    if array is None:
        ax.text(
            0.5,
            0.5,
            "unavailable",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        return

    finite = np.asarray(
        array,
        dtype=float,
    )

    finite_values = finite[
        np.isfinite(
            finite
        )
    ]

    if len(
        finite_values
    ) == 0:
        ax.text(
            0.5,
            0.5,
            "no finite pixels",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        return

    try:
        if mode == "difference":
            interval = ZScaleInterval()
            vmin, vmax = interval.get_limits(
                finite_values
            )

            norm = ImageNormalize(
                vmin=vmin,
                vmax=vmax,
                stretch=LinearStretch(),
                clip=True,
            )

        else:
            interval = PercentileInterval(
                99.0
            )

            vmin, vmax = interval.get_limits(
                finite_values
            )

            norm = ImageNormalize(
                vmin=vmin,
                vmax=vmax,
                stretch=AsinhStretch(
                    a=0.05
                ),
                clip=True,
            )

        ax.imshow(
            finite,
            origin="lower",
            cmap="gray",
            norm=norm,
        )

    except Exception:
        ax.imshow(
            finite,
            origin="lower",
            cmap="gray",
        )

## 8. Candidate quality text

In [ ]:
def fmt(
    row,
    column,
    format_spec=".3f",
):
    if column not in row.index:
        return "nan"

    value = row[
        column
    ]

    try:
        if pd.isna(
            value
        ):
            return "nan"
    except Exception:
        pass

    try:
        return format(
            value,
            format_spec,
        )
    except Exception:
        return str(
            value
        )


def get_target_band_counts_text(
    target_id,
):
    tmp = candidate_alerts[
        candidate_alerts["target_id"]
        == target_id
    ]

    if (
        "alert_properties"
        not in tmp.columns
    ):
        return "none"

    bands = (
        tmp["alert_properties"]
        .map(as_mapping)
        .map(
            lambda p: p.get(
                "lsst_diaSource_band",
                np.nan,
            )
        )
        .dropna()
        .astype(str)
    )

    if len(bands) == 0:
        return "none"

    counts = bands.value_counts()

    # Keep LSST bands in the usual order.
    band_order = [
        "u",
        "g",
        "r",
        "i",
        "z",
        "y",
    ]

    parts = [
        f"{band}:{int(counts[band])}"
        for band in band_order
        if band in counts
    ]

    # Just in case another band label exists.
    extra_bands = [
        band
        for band in counts.index
        if band not in band_order
    ]

    parts += [
        f"{band}:{int(counts[band])}"
        for band in extra_bands
    ]

    return " ".join(parts)


def flag_yn(
    value,
):
    try:
        if pd.isna(value):
            return "nan"
    except Exception:
        pass

    if isinstance(
        value,
        str,
    ):
        text = value.strip().lower()

        if text in {
            "true",
            "t",
            "yes",
            "y",
            "1",
        }:
            return "Y"

        if text in {
            "false",
            "f",
            "no",
            "n",
            "0",
            "",
        }:
            return "N"

    return (
        "Y"
        if bool(value)
        else "N"
    )


def member_loci_text(
    row,
):
    # Prefer the FoF membership already stored with the target.
    if "member_locus_ids" in row.index:
        value = row[
            "member_locus_ids"
        ]

        missing = False

        try:
            missing = bool(
                pd.isna(value)
            )
        except Exception:
            # Array/list-like values make pd.isna return an array.
            missing = False

        if not missing:
            if isinstance(
                value,
                str,
            ):
                return value

            if isinstance(
                value,
                (
                    list,
                    tuple,
                    set,
                    np.ndarray,
                    pd.Series,
                ),
            ):
                return str(
                    [
                        str(x)
                        for x in value
                    ]
                )

            return str(value)

    # Fallback: reconstruct from the candidate-alert table.
    loci = (
        candidate_alerts.loc[
            candidate_alerts["target_id"]
            == row["target_id"],
            "locus_id",
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    return str(
        sorted(loci)
    )


def quality_text(
    row,
    alert_meta=None,
):
    lines = []

    lines.append(
        "Second-Pass="
        f"{fmt(row, 'second_pass_score', '.4f')}   "
        "First-Cut="
        f"{fmt(row, 'first_cut_max_score', '.4f')}   "
        "g="
        f"{fmt(row, 'g_mag', '.2f')}   "
        "r="
        f"{fmt(row, 'r_mag', '.2f')}   "
        "i="
        f"{fmt(row, 'i_mag', '.2f')}   "
        "z="
        f"{fmt(row, 'z_mag', '.2f')}   "
        "Gaia BP-RP="
        f"{fmt(row, 'gaia_bp_rp_max', '.2f')}"
    )

    lines.append(
        "extendedness_med="
        f"{fmt(row, 'extendedness_median', '.3f')}   "
        "|ap-PSF|/|PSF|_med="
        f"{fmt(row, 'ap_psf_abs_fracdiff_median', '.3f')}   "
        "delta_e_med="
        f"{fmt(row, 'delta_e_median', '.3f')}   "
        "|delta_e|_med="
        f"{fmt(row, 'abs_delta_e_median', '.3f')}"
    )

    bands_text = (
        get_target_band_counts_text(
            row["target_id"]
        )
    )

    coord = SkyCoord(
        ra=float(row["ra"]) * u.deg,
        dec=float(row["dec"]) * u.deg,
        frame="icrs",
    )

    gal_b = (
        coord.galactic.b.deg
    )

    n_alerts = len(
        candidate_alerts[
            candidate_alerts["target_id"]
            == row["target_id"]
        ]
    )

    lines.append(
        "N_alerts="
        f"{n_alerts:d}   "
        "Lantern num_tagged_alerts="
        f"{fmt(row, 'n_tagged', '.0f')}   "
        "bands="
        f"{bands_text}   "
        "RA="
        f"{fmt(row, 'ra', '.6f')}   "
        "Dec="
        f"{fmt(row, 'dec', '.6f')}   "
        "b="
        f"{gal_b:.2f} deg"
    )

    # Put all target loci and the displayed thumbnail alert
    # on the SAME line.
    locus_line = (
        "loci="
        f"{member_loci_text(row)}"
    )

    if alert_meta is not None:
        locus_line += (
            "   thumb locus="
            f"{alert_meta.get('locus_id', '')}"
            "   thumb alert="
            f"{alert_meta.get('alert_id', '')}"
        )

        mjd = alert_meta.get(
            "mjd",
            np.nan,
        )

        if pd.notna(mjd):
            locus_line += (
                "   MJD="
                f"{float(mjd):.5f}"
            )

    lines.append(
        locus_line
    )

    flags = []

    if (
        "gaia_in_qso_candidates_any"
        in row.index
    ):
        flags.append(
            "GaiaQSO="
            + flag_yn(
                row[
                    "gaia_in_qso_candidates_any"
                ]
            )
        )

    if "milliquas_match" in row.index:
        flags.append(
            "Milliquas="
            + flag_yn(
                row[
                    "milliquas_match"
                ]
            )
        )

    if "known_lens_match" in row.index:
        flags.append(
            "KnownLens="
            + flag_yn(
                row[
                    "known_lens_match"
                ]
            )
        )

    if "known_lens_name" in row.index:
        lens_name = row[
            "known_lens_name"
        ]

        if (
            pd.notna(lens_name)
            and str(
                lens_name
            ).strip()
        ):
            flags.append(
                "LensName="
                f"{lens_name}"
            )

    if (
        "known_lens_sep_arcsec"
        in row.index
        and pd.notna(
            row[
                "known_lens_sep_arcsec"
            ]
        )
    ):
        flags.append(
            "LensSep="
            f"{row['known_lens_sep_arcsec']:.2f}\""
        )

    if flags:
        lines.append(
            "   ".join(
                flags
            )
        )

    return "\n".join(
        lines
    )


## 9. Build one six-panel candidate figure

In [ ]:
def display_thumbnail(
    ax,
    image,
    title,
):
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

    if image is None:
        ax.text(
            0.5,
            0.5,
            "unavailable",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        return

    ax.imshow(
        image,
        origin="lower",
    )
    
def make_candidate_figure(
    row,
):
    target_id = row[
        "target_id"
    ]

    print(
        f"Fetching/rendering viewer rank "
        f"{int(row['viewer_rank'])}: {target_id}"
    )

    # -------------------------
    # DP2
    # -------------------------

    irg, zir = get_dp2_irg_zir(
        row
    )

    # -------------------------
    # Legacy Survey
    # -------------------------

    try:
        legacy = get_legacy_cutout(
            row
        )
    except Exception as error:
        print(
            f"{target_id}: Legacy Survey cutout failed: {error}"
        )
        legacy = None

    # -------------------------
    # ANTARES thumbnails
    # -------------------------

    alert_meta, thumbnails = (
        get_recent_antares_thumbnails(
            target_id
        )
    )

    # -------------------------
    # Figure: 2 rows x 3 columns
    # -------------------------

    fig, axes = plt.subplots(
        2,
        3,
        figsize=FIGSIZE,
    )

    # Row 1: contextual/color images
    if irg is not None:
        axes[0, 0].imshow(
            irg,
            origin="lower",
        )
    else:
        axes[0, 0].text(
            0.5,
            0.5,
            "unavailable",
            ha="center",
            va="center",
            transform=axes[0, 0].transAxes,
        )

    axes[0, 0].set_title(
        "DP2 IRG"
    )

    if zir is not None:
        axes[0, 1].imshow(
            zir,
            origin="lower",
        )
    else:
        axes[0, 1].text(
            0.5,
            0.5,
            "unavailable",
            ha="center",
            va="center",
            transform=axes[0, 1].transAxes,
        )

    axes[0, 1].set_title(
        "DP2 ZIR"
    )

    if legacy is not None:
        axes[0, 2].imshow(
            legacy
        )
    else:
        axes[0, 2].text(
            0.5,
            0.5,
            "unavailable",
            ha="center",
            va="center",
            transform=axes[0, 2].transAxes,
        )

    axes[0, 2].set_title(
        "Legacy Survey DR11"
    )

    for ax in axes[0]:
        ax.set_xticks([])
        ax.set_yticks([])

    # Row 2: latest available ANTARES alert thumbnails
    display_thumbnail(
        axes[1, 0],
        thumbnails.get("science"),
        "ANTARES visit / science",
    )

    display_thumbnail(
        axes[1, 1],
        thumbnails.get("template"),
        "ANTARES template",
    )

    display_thumbnail(
        axes[1, 2],
        thumbnails.get("difference"),
        "ANTARES difference",
    )

    thumb_locus = "none"
    
    if alert_meta is not None:
        thumb_locus = str(
            alert_meta.get(
                "locus_id",
                "none",
            )
        )
    
    fig.suptitle(
        (
            f"viewer rank={int(row['viewer_rank'])}  "
            f"target_id={target_id}"
            f"thumb locus={thumb_locus}"
        ),
        fontsize=13,
        y=0.985,
    )

    fig.text(
        0.02,
        0.02,
        quality_text(
            row,
            alert_meta=alert_meta,
        ),
        fontsize=9,
        ha="left",
        va="bottom",
    )

    fig.subplots_adjust(
        top=0.92,
        bottom=0.17,
        left=0.04,
        right=0.99,
        hspace=0.20,
        wspace=0.08,
    )

    return (
        fig,
        alert_meta,
        {
            "dp2_irg_ok": (
                irg is not None
            ),
            "dp2_zir_ok": (
                zir is not None
            ),
            "legacy_ok": (
                legacy is not None
            ),
            "antares_template_ok": (
                thumbnails.get(
                    "template"
                )
                is not None
            ),
            "antares_science_ok": (
                thumbnails.get(
                    "science"
                )
                is not None
            ),
            "antares_difference_ok": (
                thumbnails.get(
                    "difference"
                )
                is not None
            ),
        },
    )

## 10. Show/save the selected candidates

In [ ]:
results = []

for _, row in subset.iterrows():
    target_id = row[
        "target_id"
    ]

    viewer_rank = int(
        row[
            "viewer_rank"
        ]
    )

    try:
        (
            fig,
            alert_meta,
            status,
        ) = make_candidate_figure(
            row
        )

        legacy_url = legacy_viewer_url(
            row["ra"],
            row["dec"],
        )

        pdf_path = (
            PDF_DIR
            / (
                f"{viewer_rank:04d}_"
                f"{safe_filename(target_id)}.pdf"
            )
        )

        if SAVE_PDF:
            if (
                OVERWRITE_PDF
                or not pdf_path.exists()
            ):
                fig.savefig(
                    pdf_path,
                    bbox_inches="tight",
                )

        if SHOW_IN_NOTEBOOK:
            display(
                Markdown(
                    f"**{target_id}** — "
                    f"[open Legacy Survey viewer]({legacy_url})"
                )
            )

            plt.show()

        results.append({
            "viewer_rank": viewer_rank,
            "target_id": target_id,
            "pdf_path": (
                str(pdf_path)
                if SAVE_PDF
                else ""
            ),
            "thumbnail_alert_id": (
                ""
                if alert_meta is None
                else alert_meta.get(
                    "alert_id",
                    "",
                )
            ),
            "thumbnail_mjd": (
                np.nan
                if alert_meta is None
                else alert_meta.get(
                    "mjd",
                    np.nan,
                )
            ),
            "legacy_viewer_url": legacy_url,
            **status,
        })

        plt.close(
            fig
        )

    except Exception as error:
        print(
            f"{target_id}: viewer failed: {error}"
        )

        results.append({
            "viewer_rank": viewer_rank,
            "target_id": target_id,
            "error": str(
                error
            ),
        })

results_df = pd.DataFrame(
    results
)

manifest_path = (
    OUTPUT_DIR
    / "viewer_manifest.csv"
)

results_df.to_csv(
    manifest_path,
    index=False,
)

print()
print(
    "Viewer manifest:",
    manifest_path,
)

if SAVE_PDF:
    print(
        "PDF directory:",
        PDF_DIR,
    )

display(
    results_df
)

## 11. Scaling to 100 candidates

In [ ]:
# For later use:
#
# START = 0
# N_SHOW = 100
#
# Then rerun:
#   1. the subset cell (Section 2), or redefine subset directly;
#   2. Section 10.
#
# The remote data already downloaded for earlier candidates are stored under:
#
#   lantern_candidate_cutout_viewer_v1/cache/
#
# so rerunning does not re-download them.
#
# For a staged run, a safer pattern is:
#
#   START = 0;   N_SHOW = 20
#   START = 20;  N_SHOW = 20
#   START = 40;  N_SHOW = 20
#   ...
#
# The DP2 helper is deliberately serial and sleeps between new SODA requests.
# If HTTP 429 still occurs, it obeys the server's Retry-After message and retries.

## Notes

### Candidate quality information

The figure text includes, when available:

- **Second-Pass score**;
- First-Cut maximum score;
- `i`-band diaObject magnitude;
- Gaia BP-RP;
- median extendedness;
- median `|apFlux - psfFlux| / |psfFlux|`;
- median `delta_e` and median `|delta_e|`;
- number of LSST alerts;
- known-lens match information;
- the ANTARES alert ID and MJD used for thumbnails.

### ANTARES thumbnails

The notebook uses the current ANTARES client interface:

```python
from antares_client.search import get_thumbnails
get_thumbnails("lsst:<alert_id>")
```

It tries the newest alert first and, if necessary, a few earlier alerts.

### Rate limiting

DP2 deep-coadd images are the expensive part. The notebook therefore:

- retrieves serially;
- pauses between newly downloaded bands;
- catches HTTP 429;
- uses the service's reported retry interval when available;
- caches each band as a local `.npy` file.

The Legacy JPEG and ANTARES thumbnails are also cached locally.